# Ev Eşyası Tespiti: YOLOv8s + COCO 2017 (9 sınıf)

Bu not defteri ödevin tüm adımlarını Google Colab'da (T4 GPU) çalıştırır:

1. Ortam kurulumu
2. COCO 2017'den 9 sınıflık alt kümenin hazırlanması (train / val / test)
3. YOLOv8s fine-tuning
4. Test kümesinde değerlendirme (Precision, Recall, F1, mAP, IoU, süre)
5. Telefon fotoğraflarıyla test
6. Gradio uygulaması

**Nasıl çalıştırılır:**
1. *Çalışma zamanı → Çalışma zamanı türünü değiştir → T4 GPU*
2. *Çalışma zamanı → Tümünü çalıştır*, Drive erişim izni isteyince onaylayın.
3. Sekmeyi açık bırakın (~2 saat). Tüm çıktılar Drive'da `MyDrive/dlodev2` klasörüne yazılır.

## 1. Ortam kurulumu

In [ ]:
import torch
assert torch.cuda.is_available(), "GPU yok! Çalışma zamanı türünü T4 GPU yapın."
!nvidia-smi
from google.colab import drive
drive.mount('/content/drive')

PROJECT_DIR = '/content/drive/MyDrive/dlodev2'
DATA_DIR = '/content/coco9'   # Veri yerel diskte tutulur (Drive'dan çok daha hızlı)
%cd {PROJECT_DIR}
!pip install -q -r requirements.txt

## 2. Veri setinin hazırlanması

Anotasyonlar (~241 MB) ve yalnızca gereken görüntüler indirilir (~1,5 GB, yaklaşık 5-10 dakika).
Colab oturumu kapanırsa `/content` silinir; bu hücreyi yeniden çalıştırmanız gerekir.

In [ ]:
!python prepare_coco_subset.py --out {DATA_DIR} --max-train 8000 --negatives 800

## 3. Eğitim (YOLOv8s fine-tuning)

Çıktılar Drive'a (`runs/yolov8s_coco9`) yazılır; oturum kopsa da kaybolmaz.
T4 GPU ile 8800 görüntü × 20 epoch yaklaşık 1-1,5 saat sürer.

In [ ]:
!python train.py --data {DATA_DIR}/coco9.yaml

Eğitim yarıda kalırsa: önce 1. ve 2. hücreleri tekrar çalıştırın, sonra aşağıdaki satırın başındaki `#` işaretini kaldırın.

In [ ]:
# !python train.py --resume {PROJECT_DIR}/runs/yolov8s_coco9/weights/last.pt

In [ ]:
# Eğitim eğrileri (kayıp ve mAP), sınıf dağılımı ve etiketli örnek eğitim grubu
from IPython.display import Image, display
RUN = f'{PROJECT_DIR}/runs/yolov8s_coco9'
for name in ['results.png', 'labels.jpg', 'train_batch0.jpg']:
    display(Image(f'{RUN}/{name}', width=1000))

## 4. Test kümesinde değerlendirme

In [ ]:
!python evaluate.py --data {DATA_DIR}/coco9.yaml --cpu-timing 50

In [ ]:
import glob
from IPython.display import Image, Markdown, display
display(Markdown(open('results/metrics_table.md', encoding='utf-8').read()))
for name in ['confusion_matrix_normalized', '*PR_curve']:
    for p in glob.glob(f'results/test_eval/{name}.png'):
        display(Image(p, width=800))

## 5. Telefon fotoğraflarıyla test (Ödev Bölüm 8)

In [ ]:
!python predict_test_photos.py
from IPython.display import Image, Markdown, display
display(Markdown(open('results/test_photos/sonuclar.md', encoding='utf-8').read()))
display(Image('results/test_photos/grid.jpg', width=1100))

## 6. Gradio uygulaması

`--share` ile herkese açık geçici bir link (72 saat geçerli) oluşturulur. Durdurmak için hücreyi durdurun.

In [ ]:
!python app.py --share